In [1]:
!pip install semopy

In [24]:
# ================================================================
# SEM Model with manual bootstrap (works with all semopy versions)
# HBA1C → LTEMPPOLE → TMTA (Adjusted for AGE, SEX, SBP, DBP)
# ================================================================

import pandas as pd
import numpy as np
from semopy import Model, calc_stats
from sklearn.preprocessing import StandardScaler
import os
import time

# ----------------------- USER INPUTS ----------------------------
FILE = "Multiclass data.xlsx"
DV = "TMTB"
MED = "LTEMPPOLE"
IV = "HBA1C"
COVARS = ["AGE", "SEX", "SBP", "DBP"]
N_BOOT = 5000   # desired number of successful bootstrap replicates
MAX_ATTEMPTS = N_BOOT * 5  # max attempts (to allow for failed fits)
OUTPUT_DIR = "SEM_Results_LTEMPPOLE"
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_SEED = 12345

# ----------------------- LOAD & STANDARDIZE DATA ----------------
data = pd.read_excel(FILE)
cols = [DV, MED, IV] + COVARS
df_full = data[cols].dropna().copy().reset_index(drop=True)

# Standardize numeric variables (fit on observed rows)
scaler = StandardScaler()
df = df_full.copy()
df[df.columns] = scaler.fit_transform(df[df.columns])

print(f"✅ Data loaded and standardized: {df.shape[0]} rows, {df.shape[1]} variables")

# ----------------------- MODEL SPECIFICATION --------------------
model_desc = f"""
{MED} ~ a*{IV} + {' + '.join(COVARS)}
{DV} ~ b*{MED} + c_prime*{IV} + {' + '.join(COVARS)}
"""

print("\n🔧 SEM Model:\n", model_desc)

# ----------------------- FIT ORIGINAL MODEL ---------------------
model = Model(model_desc)
try:
    model.fit(df)
except Exception as e:
    print("⚠️ Warning: initial model fit raised an exception:", e)

# Get parameter estimates from the original fit
estimates = model.inspect(std_est=True)
stats = calc_stats(model)

# helper to get parameter estimate by lval and rval
def get_param(est_df, lval, rval):
    sel = est_df.loc[(est_df["lval"] == lval) & (est_df["rval"] == rval)]
    if sel.shape[0] == 0:
        raise KeyError(f"Parameter {lval}~{rval} not found in estimates.")
    return sel["Estimate"].values[0]

try:
    a = get_param(estimates, MED, IV)
    b = get_param(estimates, DV, MED)
    c_prime = get_param(estimates, DV, IV)
except KeyError as e:
    raise RuntimeError("Required path missing from model estimates: " + str(e))

indirect = a * b
total = c_prime + indirect

print("\n=== Original model path estimates (standardized) ===")
print(f"a (IV→MED)       : {a:.4f}")
print(f"b (MED→DV)       : {b:.4f}")
print(f"c' (IV→DV direct): {c_prime:.4f}")
print(f"Indirect (a*b)   : {indirect:.4f}")
print(f"Total            : {total:.4f}")

# ----------------------- MANUAL BOOTSTRAP -----------------------
rng = np.random.default_rng(RANDOM_SEED)
n_obs = df.shape[0]

# We'll collect parameter estimates per parameter name "lval~rval"
param_names = None  # will be set after the first successful bootstrap
param_samples = dict()
successful = 0
attempts = 0
start_time = time.time()
print(f"\n🔁 Starting manual bootstrap: aim for {N_BOOT} successful replications...")

while successful < N_BOOT and attempts < MAX_ATTEMPTS:
    attempts += 1
    # sample indices with replacement
    idx = rng.integers(low=0, high=n_obs, size=n_obs)
    df_boot = df.iloc[idx].reset_index(drop=True)

    # fit a fresh model on the bootstrap sample
    try:
        model_b = Model(model_desc)
        model_b.fit(df_boot)
        est_b = model_b.inspect(std_est=True)
        # Build parameter name vector if first success
        if param_names is None:
            param_names = [f"{row['lval']}~{row['rval']}" for _, row in est_b.iterrows()]
            # initialize storage lists
            for pn in param_names:
                param_samples[pn] = []
        # append estimates in the same order as param_names
        # create a mapping for safety
        est_map = {f"{row['lval']}~{row['rval']}": row["Estimate"] for _, row in est_b.iterrows()}
        # If some parameter missing in this bootstrap (rare), set NaN
        for pn in param_names:
            param_samples[pn].append(est_map.get(pn, np.nan))
        successful += 1
        if successful % 50 == 0 or successful <= 5:
            print(f"  • successful bootstraps: {successful} (attempts: {attempts})")
    except Exception as e:
        # Fit failed for this bootstrap; skip it
        if attempts <= 10 or attempts % 50 == 0:
            print(f"  - bootstrap attempt {attempts} failed: {e}")
        continue

elapsed = time.time() - start_time
print(f"\n🔚 Bootstrap finished: {successful} successful out of {attempts} attempts in {elapsed:.1f}s")

if successful == 0:
    raise RuntimeError("Bootstrap failed: no successful bootstrap fits. Check model stability or reduce N_BOOT.")

# Convert lists to arrays (shape: successful x parameters)
for pn in param_names:
    param_samples[pn] = np.array(param_samples[pn], dtype=float)

# ----------------------- COMPUTE BOOTSTRAP DISTRIBUTIONS ----------
# Get arrays for a and b (use parameter name format used earlier)
name_a = f"{MED}~{IV}"
name_b = f"{DV}~{MED}"

if name_a not in param_samples or name_b not in param_samples:
    raise RuntimeError("Bootstrap did not capture required parameters for a or b.")

a_boot = param_samples[name_a]
b_boot = param_samples[name_b]

# Remove any NaN rows (where either a or b is NaN)
valid_mask = (~np.isnan(a_boot)) & (~np.isnan(b_boot))
a_boot_valid = a_boot[valid_mask]
b_boot_valid = b_boot[valid_mask]
ab_boot = a_boot_valid * b_boot_valid

if ab_boot.size == 0:
    lower, upper = np.nan, np.nan
    print("⚠️ No valid a*b bootstrap samples found.")
else:
    lower, upper = np.percentile(ab_boot, [2.5, 97.5])

# Also compute bootstrap SEs and CIs for a, b, c' if desired
def bootstrap_ci(arr, alpha=0.05):
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return (np.nan, np.nan)
    return np.percentile(arr, [100*alpha/2, 100*(1-alpha/2)])

a_ci = bootstrap_ci(a_boot_valid)
b_ci = bootstrap_ci(b_boot_valid)
c_name = f"{DV}~{IV}"
c_boot = param_samples.get(c_name, np.array([np.nan]*successful))
c_boot_valid = c_boot[~np.isnan(c_boot)]
c_ci = bootstrap_ci(c_boot_valid)

# ----------------------- DISPLAY BOOTSTRAP RESULTS ----------------
print("\n=== Bootstrap summary ===")
print(f"Bootstrap draws used (successful): {a_boot_valid.size}")
print(f"a (orig): {a:.4f}, bootstrap 95% CI: [{a_ci[0]:.4f}, {a_ci[1]:.4f}]")
print(f"b (orig): {b:.4f}, bootstrap 95% CI: [{b_ci[0]:.4f}, {b_ci[1]:.4f}]")
print(f"c' (orig): {c_prime:.4f}, bootstrap 95% CI: [{c_ci[0]:.4f}, {c_ci[1]:.4f}]")
print(f"Indirect (a*b) original: {indirect:.4f}")
print(f"Indirect (a*b) bootstrap 95% CI: [{lower:.4f}, {upper:.4f}]")

# ----------------------- EXPORT RESULTS -------------------------
# Save path estimates, fit stats, and bootstrap summaries
estimates_out = estimates.copy()
estimates_out["Bootstrap_Mean"] = np.nan
estimates_out["Bootstrap_SE"] = np.nan
estimates_out["Bootstrap_L95"] = np.nan
estimates_out["Bootstrap_U95"] = np.nan

# fill bootstrap summaries for parameters we have
for i, row in estimates_out.iterrows():
    pn = f"{row['lval']}~{row['rval']}"
    if pn in param_samples:
        arr = param_samples[pn]
        arr_valid = arr[~np.isnan(arr)]
        if arr_valid.size > 0:
            estimates_out.at[i, "Bootstrap_Mean"] = np.nanmean(arr_valid)
            estimates_out.at[i, "Bootstrap_SE"] = np.nanstd(arr_valid, ddof=1)
            ci_low, ci_up = bootstrap_ci(arr_valid)
            estimates_out.at[i, "Bootstrap_L95"] = ci_low
            estimates_out.at[i, "Bootstrap_U95"] = ci_up

# Save to Excel and text
excel_path = os.path.join(OUTPUT_DIR, "SEM_Results_withBootstrap.xlsx")
txt_path = os.path.join(OUTPUT_DIR, "SEM_Summary_withBootstrap.txt")

with pd.ExcelWriter(excel_path) as writer:
    estimates_out.to_excel(writer, sheet_name="Path_Estimates", index=False)
    pd.DataFrame(stats, index=[0]).to_excel(writer, sheet_name="Fit_Indices", index=False)
    pd.DataFrame({
        "a": [a],
        "b": [b],
        "c_prime": [c_prime],
        "Indirect": [indirect],
        "Total": [total],
        "Indirect_L95": [lower],
        "Indirect_U95": [upper],
        "Bootstrap_draws": [a_boot_valid.size],
        "Bootstrap_attempts": [attempts],
    }).to_excel(writer, sheet_name="Effects_Summary", index=False)

with open(txt_path, "w") as f:
    f.write("=== SEM MODEL WITH MANUAL BOOTSTRAP SUMMARY ===\n")
    f.write(f"Model: {IV} -> {MED} -> {DV} (covariates: {', '.join(COVARS)})\n")
    f.write(f"Data rows used: {n_obs}\n")
    f.write(f"Bootstrap successful draws: {a_boot_valid.size}\n\n")
    f.write("Original path estimates:\n")
    f.write(f"a (IV->MED): {a:.4f}\n")
    f.write(f"b (MED->DV): {b:.4f}\n")
    f.write(f"c' (IV->DV): {c_prime:.4f}\n")
    f.write(f"Indirect: {indirect:.4f}\n")
    f.write(f"Indirect 95% CI (bootstrap): [{lower:.4f}, {upper:.4f}]\n\n")
    f.write("Bootstrap notes:\n")
    f.write(f"Attempts made: {attempts}\n")
    f.write(f"Successful fits: {successful}\n")

print(f"\n✅ Results exported to:\n  • {excel_path}\n  • {txt_path}")
print("🎯 Manual bootstrap SEM completed.")


✅ Data loaded and standardized: 200 rows, 7 variables

🔧 SEM Model:
 
LTEMPPOLE ~ a*HBA1C + AGE + SEX + SBP + DBP
TMTB ~ b*LTEMPPOLE + c_prime*HBA1C + AGE + SEX + SBP + DBP


=== Original model path estimates (standardized) ===
a (IV→MED)       : -0.1112
b (MED→DV)       : 0.0671
c' (IV→DV direct): 0.1775
Indirect (a*b)   : -0.0075
Total            : 0.1700

🔁 Starting manual bootstrap: aim for 5000 successful replications...
  • successful bootstraps: 1 (attempts: 1)
  • successful bootstraps: 2 (attempts: 2)
  • successful bootstraps: 3 (attempts: 3)
  • successful bootstraps: 4 (attempts: 4)
  • successful bootstraps: 5 (attempts: 5)
  • successful bootstraps: 50 (attempts: 50)
  • successful bootstraps: 100 (attempts: 100)
  • successful bootstraps: 150 (attempts: 150)
  • successful bootstraps: 200 (attempts: 200)
  • successful bootstraps: 250 (attempts: 250)
  • successful bootstraps: 300 (attempts: 300)
  • successful bootstraps: 350 (attempts: 350)
  • successful bootstraps: 4

In [2]:
# ================================================================
# CONSOLIDATED FINAL VERSION
# ================================================================

import pandas as pd
import numpy as np
from semopy import Model, calc_stats
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import os
import time
import warnings

warnings.filterwarnings("ignore")


# ================================================================
# 1. USER INPUTS
# ================================================================

FILE = "Multiclass data.xlsx"

DV = "TMTB"
MED = "LTEMPPOLE"
IV = "HBA1C"

COVARS = ["AGE", "SEX", "SBP", "DBP"]

# Reviewer 2 adjustment
ADJUSTMENT_VAR = "TMTA"

N_BOOT = 5000
MAX_ATTEMPTS = N_BOOT * 5

OUTPUT_DIR = "SEM_Results_LTEMPPOLE_TMTB_Adjusted_TMA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 12345


# ================================================================
# 2. LOAD DATA
# ================================================================

data = pd.read_excel(FILE)

print("=" * 80)
print("TMT-A-ADJUSTED SEM WITH MANUAL BOOTSTRAP")
print("=" * 80)

print(f"\nDataset: {FILE}")
print(f"Original dataset shape: {data.shape}")


# ================================================================
# 3. REQUIRED VARIABLES
# ================================================================

required_cols = [
    DV,
    MED,
    IV,
    ADJUSTMENT_VAR
] + COVARS

missing_cols = [
    col for col in required_cols
    if col not in data.columns
]

if missing_cols:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(missing_cols)
    )

print("\nRequired variables:")
for col in required_cols:
    print(f"  ✓ {col}")


# ================================================================
# 4. COMPLETE-CASE DATASET
# ================================================================

df_full = (
    data[required_cols]
    .copy()
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
    .reset_index(drop=True)
)

n_obs = len(df_full)

print("\nComplete-case analysis:")
print(f"  Original N       : {len(data)}")
print(f"  Complete-case N  : {n_obs}")
print(f"  Excluded         : {len(data) - n_obs}")


if n_obs < 50:
    print(
        "\n⚠️ WARNING: Complete-case sample size is relatively small."
    )


# ================================================================
# 5. STANDARDIZATION
#
# Standardize after complete-case selection.
#
# This makes the SEM path coefficients standardized because all
# variables entering the model are z-standardized.
# ================================================================

scaler = StandardScaler()

df = pd.DataFrame(
    scaler.fit_transform(df_full),
    columns=df_full.columns
)

print("\nVariables standardized:")
print("  " + ", ".join(df.columns))


# ================================================================
# 6. SEM MODEL SPECIFICATION
#
# Mediator:
#   LTEMPPOLE <- HBA1C + AGE + SEX + SBP + DBP
#
# Outcome:
#   TMTB <- LTEMPPOLE + HBA1C + TMTA + AGE + SEX + SBP + DBP
#
# TMTA is specifically included in the TMTB equation to account
# for processing-speed/motor-performance components.
# ================================================================

model_desc = f"""
{MED} ~ a*{IV} + AGE + SEX + SBP + DBP

{DV} ~ b*{MED} + c_prime*{IV} + {ADJUSTMENT_VAR} + AGE + SEX + SBP + DBP
"""

print("\n" + "=" * 80)
print("SEM MODEL")
print("=" * 80)
print(model_desc)


# ================================================================
# 7. FIT ORIGINAL SEM
# ================================================================

model = Model(model_desc)

try:
    model.fit(df)
except Exception as e:
    raise RuntimeError(
        f"Initial SEM fitting failed:\n{e}"
    )


# ================================================================
# 8. PARAMETER ESTIMATES
# ================================================================

estimates = model.inspect(std_est=True)

print("\n" + "=" * 80)
print("SEM PARAMETER ESTIMATES")
print("=" * 80)

print(estimates.to_string(index=False))


# ================================================================
# 9. FIT STATISTICS
# ================================================================

stats = calc_stats(model)

print("\n" + "=" * 80)
print("MODEL FIT")
print("=" * 80)

print(stats.to_string())


# ================================================================
# 10. HELPER FUNCTION
# ================================================================

def get_param(est_df, lval, rval):

    sel = est_df.loc[
        (est_df["lval"] == lval) &
        (est_df["rval"] == rval)
    ]

    if sel.shape[0] == 0:
        raise KeyError(
            f"Parameter {lval} ~ {rval} not found."
        )

    return sel["Estimate"].values[0]


# ================================================================
# 11. EXTRACT KEY PATHS
# ================================================================

try:

    a = get_param(
        estimates,
        MED,
        IV
    )

    b = get_param(
        estimates,
        DV,
        MED
    )

    c_prime = get_param(
        estimates,
        DV,
        IV
    )

    tmt_a_path = get_param(
        estimates,
        DV,
        ADJUSTMENT_VAR
    )

except KeyError as e:

    raise RuntimeError(
        "Required SEM path missing: " + str(e)
    )


# ================================================================
# 12. INDIRECT AND TOTAL EFFECTS
# ================================================================

indirect = a * b
total = c_prime + indirect


print("\n" + "=" * 80)
print("KEY SEM PATHS")
print("=" * 80)

print(f"a  HbA1c → LTEMPPOLE       : {a:.4f}")
print(f"b  LTEMPPOLE → TMTB        : {b:.4f}")
print(f"c' HbA1c → TMTB            : {c_prime:.4f}")
print(f"   TMTA → TMTB              : {tmt_a_path:.4f}")
print(f"Indirect (a × b)            : {indirect:.4f}")
print(f"Total effect                : {total:.4f}")


# ================================================================
# 13. TEST STATUS × TMTA INTERACTION
#
# This uses the original STATUS variable and is separate from
# the SEM. It checks whether the TMT-A/TMT-B relationship differs
# across the four diagnostic groups.
# ================================================================

print("\n" + "=" * 80)
print("ANCOVA ASSUMPTION CHECK")
print("STATUS × TMTA interaction")
print("=" * 80)

interaction_df = (
    data[
        ["STATUS", "TMTB", "TMTA"]
    ]
    .apply(
        lambda x: pd.to_numeric(x, errors="coerce")
        if x.name != "STATUS" else x
    )
    .dropna()
    .copy()
)

interaction_df["STATUS"] = interaction_df["STATUS"].astype("category")

interaction_model_reduced = smf.ols(
    "TMTB ~ TMTA + C(STATUS)",
    data=interaction_df
).fit()

interaction_model_full = smf.ols(
    "TMTB ~ TMTA * C(STATUS)",
    data=interaction_df
).fit()

interaction_anova = anova_lm(
    interaction_model_reduced,
    interaction_model_full
)

print("\nReduced model:")
print("TMTB ~ TMTA + STATUS")

print("\nFull model:")
print("TMTB ~ TMTA * STATUS")

print("\nInteraction test:")
print(interaction_anova.to_string())

interaction_p = interaction_anova["Pr(>F)"].iloc[-1]

print(
    f"\nSTATUS × TMTA interaction p-value: "
    f"{interaction_p:.6g}"
)

if interaction_p < 0.05:
    print(
        "\n⚠️ WARNING: The STATUS × TMTA interaction is significant."
    )
    print(
        "The common-slope ANCOVA assumption is not supported."
    )
    print(
        "Interpret the TMT-A-adjusted group effect cautiously."
    )
else:
    print(
        "\n✓ STATUS × TMTA interaction is not significant."
    )
    print(
        "The common-slope assumption is supported."
    )


# ================================================================
# 14. MANUAL BOOTSTRAP
# ================================================================

rng = np.random.default_rng(RANDOM_SEED)

param_names = None
param_samples = {}

successful = 0
attempts = 0

start_time = time.time()

print("\n" + "=" * 80)
print(f"MANUAL BOOTSTRAP: {N_BOOT} SUCCESSFUL REPLICATES")
print("=" * 80)


while successful < N_BOOT and attempts < MAX_ATTEMPTS:

    attempts += 1

    # ------------------------------------------------------------
    # Bootstrap sample
    # ------------------------------------------------------------

    idx = rng.integers(
        low=0,
        high=n_obs,
        size=n_obs
    )

    df_boot = (
        df.iloc[idx]
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------
    # Fit bootstrap SEM
    # ------------------------------------------------------------

    try:

        model_b = Model(model_desc)

        model_b.fit(df_boot)

        est_b = model_b.inspect(std_est=True)

        # --------------------------------------------------------
        # Establish parameter list from first successful fit
        # --------------------------------------------------------

        if param_names is None:

            param_names = [
                f"{row['lval']}~{row['rval']}"
                for _, row in est_b.iterrows()
            ]

            for pn in param_names:
                param_samples[pn] = []

        # --------------------------------------------------------
        # Parameter mapping
        # --------------------------------------------------------

        est_map = {
            f"{row['lval']}~{row['rval']}":
            row["Estimate"]
            for _, row in est_b.iterrows()
        }

        for pn in param_names:

            param_samples[pn].append(
                est_map.get(pn, np.nan)
            )

        successful += 1

        if successful <= 5 or successful % 100 == 0:

            print(
                f"  Successful: {successful:4d} | "
                f"Attempts: {attempts:4d}"
            )

    except Exception as e:

        if attempts <= 10 or attempts % 100 == 0:

            print(
                f"  Bootstrap attempt {attempts} failed: "
                f"{str(e)[:120]}"
            )

        continue


elapsed = time.time() - start_time

print("\nBootstrap completed.")
print(f"Successful fits : {successful}")
print(f"Total attempts   : {attempts}")
print(f"Elapsed time     : {elapsed:.1f} seconds")


if successful == 0:

    raise RuntimeError(
        "No successful bootstrap SEM fits."
    )


# ================================================================
# 15. CONVERT BOOTSTRAP RESULTS TO ARRAYS
# ================================================================

for pn in param_names:

    param_samples[pn] = np.asarray(
        param_samples[pn],
        dtype=float
    )


# ================================================================
# 16. HELPER: BOOTSTRAP CI
# ================================================================

def bootstrap_ci(arr, alpha=0.05):

    arr = arr[
        ~np.isnan(arr)
    ]

    if arr.size == 0:
        return np.nan, np.nan

    return np.percentile(
        arr,
        [
            100 * alpha / 2,
            100 * (1 - alpha / 2)
        ]
    )


# ================================================================
# 17. EXTRACT BOOTSTRAP PATHS
# ================================================================

name_a = f"{MED}~{IV}"
name_b = f"{DV}~{MED}"
name_c = f"{DV}~{IV}"
name_tmt_a = f"{DV}~{ADJUSTMENT_VAR}"


a_boot = param_samples[name_a]
b_boot = param_samples[name_b]
c_boot = param_samples[name_c]
tmt_a_boot = param_samples[name_tmt_a]


# Remove failed parameter estimates
valid_a = ~np.isnan(a_boot)
valid_b = ~np.isnan(b_boot)
valid_c = ~np.isnan(c_boot)
valid_tmt_a = ~np.isnan(tmt_a_boot)


a_boot_valid = a_boot[valid_a]
b_boot_valid = b_boot[valid_b]
c_boot_valid = c_boot[valid_c]
tmt_a_boot_valid = tmt_a_boot[valid_tmt_a]


# ================================================================
# 18. BOOTSTRAP INDIRECT EFFECT
# ================================================================

valid_ab = (
    ~np.isnan(a_boot) &
    ~np.isnan(b_boot)
)

ab_boot = (
    a_boot[valid_ab] *
    b_boot[valid_ab]
)

if len(ab_boot) > 0:

    indirect_lower, indirect_upper = (
        np.percentile(
            ab_boot,
            [2.5, 97.5]
        )
    )

else:

    indirect_lower = np.nan
    indirect_upper = np.nan


# ================================================================
# 19. BOOTSTRAP TOTAL EFFECT
# ================================================================

valid_total = (
    ~np.isnan(a_boot) &
    ~np.isnan(b_boot) &
    ~np.isnan(c_boot)
)

total_boot = (
    a_boot[valid_total] *
    b_boot[valid_total] +
    c_boot[valid_total]
)

if len(total_boot) > 0:

    total_lower, total_upper = (
        np.percentile(
            total_boot,
            [2.5, 97.5]
        )
    )

else:

    total_lower = np.nan
    total_upper = np.nan


# ================================================================
# 20. CIs FOR INDIVIDUAL PATHS
# ================================================================

a_ci = bootstrap_ci(a_boot_valid)
b_ci = bootstrap_ci(b_boot_valid)
c_ci = bootstrap_ci(c_boot_valid)
tmt_a_ci = bootstrap_ci(tmt_a_boot_valid)


# ================================================================
# 21. BOOTSTRAP SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("BOOTSTRAP RESULTS")
print("=" * 80)

print(
    f"a  HbA1c → LTEMPPOLE: "
    f"{a:.4f} "
    f"[95% CI {a_ci[0]:.4f}, {a_ci[1]:.4f}]"
)

print(
    f"b  LTEMPPOLE → TMTB: "
    f"{b:.4f} "
    f"[95% CI {b_ci[0]:.4f}, {b_ci[1]:.4f}]"
)

print(
    f"c' HbA1c → TMTB: "
    f"{c_prime:.4f} "
    f"[95% CI {c_ci[0]:.4f}, {c_ci[1]:.4f}]"
)

print(
    f"TMTA → TMTB: "
    f"{tmt_a_path:.4f} "
    f"[95% CI {tmt_a_ci[0]:.4f}, "
    f"{tmt_a_ci[1]:.4f}]"
)

print(
    f"Indirect effect: "
    f"{indirect:.4f} "
    f"[95% CI {indirect_lower:.4f}, "
    f"{indirect_upper:.4f}]"
)

print(
    f"Total effect: "
    f"{total:.4f} "
    f"[95% CI {total_lower:.4f}, "
    f"{total_upper:.4f}]"
)


# ================================================================
# 22. EXPORT PATH ESTIMATES
# ================================================================

estimates_out = estimates.copy()

estimates_out["Bootstrap_Mean"] = np.nan
estimates_out["Bootstrap_SE"] = np.nan
estimates_out["Bootstrap_L95"] = np.nan
estimates_out["Bootstrap_U95"] = np.nan


for i, row in estimates_out.iterrows():

    pn = (
        f"{row['lval']}~{row['rval']}"
    )

    if pn in param_samples:

        arr = param_samples[pn]

        arr_valid = arr[
            ~np.isnan(arr)
        ]

        if arr_valid.size > 0:

            estimates_out.at[
                i,
                "Bootstrap_Mean"
            ] = np.mean(arr_valid)

            estimates_out.at[
                i,
                "Bootstrap_SE"
            ] = np.std(
                arr_valid,
                ddof=1
            )

            ci_low, ci_up = bootstrap_ci(
                arr_valid
            )

            estimates_out.at[
                i,
                "Bootstrap_L95"
            ] = ci_low

            estimates_out.at[
                i,
                "Bootstrap_U95"
            ] = ci_up


# ================================================================
# 23. EFFECTS SUMMARY TABLE
# ================================================================

effects_summary = pd.DataFrame({

    "Effect": [
        "HbA1c → LTEMPPOLE",
        "LTEMPPOLE → TMTB",
        "HbA1c → TMTB (direct)",
        "TMTA → TMTB",
        "Indirect effect (a×b)",
        "Total effect"
    ],

    "Estimate": [
        a,
        b,
        c_prime,
        tmt_a_path,
        indirect,
        total
    ],

    "CI_L95": [
        a_ci[0],
        b_ci[0],
        c_ci[0],
        tmt_a_ci[0],
        indirect_lower,
        total_lower
    ],

    "CI_U95": [
        a_ci[1],
        b_ci[1],
        c_ci[1],
        tmt_a_ci[1],
        indirect_upper,
        total_upper
    ]
})


# ================================================================
# 24. ASSUMPTION CHECK TABLE
# ================================================================

interaction_summary = pd.DataFrame({

    "Test": [
        "STATUS × TMTA interaction"
    ],

    "F": [
        interaction_anova["F"].iloc[-1]
    ],

    "p_value": [
        interaction_p
    ],

    "Interpretation": [
        "Significant"
        if interaction_p < 0.05
        else "Not significant"
    ]
})


# ================================================================
# 25. MODEL FIT TABLE
# ================================================================

fit_stats = pd.DataFrame(stats)


# ================================================================
# 26. EXPORT TO EXCEL
# ================================================================

excel_path = os.path.join(
    OUTPUT_DIR,
    "SEM_TMTB_Adjusted_for_TMTA.xlsx"
)


with pd.ExcelWriter(
    excel_path
) as writer:

    estimates_out.to_excel(
        writer,
        sheet_name="Path_Estimates",
        index=False
    )

    fit_stats.to_excel(
        writer,
        sheet_name="Fit_Indices",
        index=False
    )

    effects_summary.to_excel(
        writer,
        sheet_name="Effects_Summary",
        index=False
    )

    interaction_summary.to_excel(
        writer,
        sheet_name="TMTA_Assumption_Check",
        index=False
    )


# ================================================================
# 27. TEXT SUMMARY
# ================================================================

txt_path = os.path.join(
    OUTPUT_DIR,
    "SEM_TMTB_Adjusted_for_TMTA_Summary.txt"
)


with open(
    txt_path,
    "w"
) as f:

    f.write(
        "============================================================\n"
    )

    f.write(
        "TMT-A-ADJUSTED SEM WITH MANUAL BOOTSTRAP\n"
    )

    f.write(
        "============================================================\n\n"
    )

    f.write(
        f"Dataset: {FILE}\n"
    )

    f.write(
        f"Complete-case N: {n_obs}\n"
    )

    f.write(
        f"Bootstrap successful fits: {successful}\n"
    )

    f.write(
        f"Bootstrap attempts: {attempts}\n\n"
    )

    f.write(
        "MODEL:\n"
    )

    f.write(
        "LTEMPPOLE ~ HBA1C + AGE + SEX + SBP + DBP\n"
    )

    f.write(
        "TMTB ~ LTEMPPOLE + HBA1C + TMTA + AGE + SEX + SBP + DBP\n\n"
    )

    f.write(
        "KEY PATHS:\n"
    )

    f.write(
        f"HbA1c -> LTEMPPOLE: "
        f"{a:.4f} "
        f"[{a_ci[0]:.4f}, {a_ci[1]:.4f}]\n"
    )

    f.write(
        f"LTEMPPOLE -> TMTB: "
        f"{b:.4f} "
        f"[{b_ci[0]:.4f}, {b_ci[1]:.4f}]\n"
    )

    f.write(
        f"HbA1c -> TMTB direct: "
        f"{c_prime:.4f} "
        f"[{c_ci[0]:.4f}, {c_ci[1]:.4f}]\n"
    )

    f.write(
        f"TMTA -> TMTB: "
        f"{tmt_a_path:.4f} "
        f"[{tmt_a_ci[0]:.4f}, {tmt_a_ci[1]:.4f}]\n"
    )

    f.write(
        f"Indirect effect: "
        f"{indirect:.4f} "
        f"[{indirect_lower:.4f}, "
        f"{indirect_upper:.4f}]\n"
    )

    f.write(
        f"Total effect: "
        f"{total:.4f} "
        f"[{total_lower:.4f}, "
        f"{total_upper:.4f}]\n\n"
    )

    f.write(
        "ANCOVA ASSUMPTION:\n"
    )

    f.write(
        f"STATUS x TMTA interaction p-value: "
        f"{interaction_p:.6g}\n"
    )

    if interaction_p < 0.05:

        f.write(
            "Interpretation: Significant interaction; "
            "common-slope assumption not supported.\n"
        )

    else:

        f.write(
            "Interpretation: Nonsignificant interaction; "
            "common-slope assumption supported.\n"
        )


print("\n" + "=" * 80)
print("EXPORT COMPLETED")
print("=" * 80)

print(f"\nExcel:")
print(excel_path)

print(f"\nText summary:")
print(txt_path)

print("\n✓ TMT-A adjustment incorporated directly into SEM.")
print("✓ No artificial patient-level adjusted TMT-B variable created.")
print("✓ Bootstrap indirect effect calculated.")
print("✓ STATUS × TMTA assumption check included.")

TMT-A-ADJUSTED SEM WITH MANUAL BOOTSTRAP

Dataset: Multiclass data.xlsx
Original dataset shape: (200, 17)

Required variables:
  ✓ TMTB
  ✓ LTEMPPOLE
  ✓ HBA1C
  ✓ TMTA
  ✓ AGE
  ✓ SEX
  ✓ SBP
  ✓ DBP

Complete-case analysis:
  Original N       : 200
  Complete-case N  : 200
  Excluded         : 0

Variables standardized:
  TMTB, LTEMPPOLE, HBA1C, TMTA, AGE, SEX, SBP, DBP

SEM MODEL

LTEMPPOLE ~ a*HBA1C + AGE + SEX + SBP + DBP

TMTB ~ b*LTEMPPOLE + c_prime*HBA1C + TMTA + AGE + SEX + SBP + DBP


SEM PARAMETER ESTIMATES
     lval op      rval  Estimate  Est. Std  Std. Err   z-value      p-value
LTEMPPOLE  ~     HBA1C -0.111166 -0.111160  0.075563 -1.471171 1.412449e-01
LTEMPPOLE  ~       AGE  0.274088  0.274074  0.067586  4.055368 5.005538e-05
LTEMPPOLE  ~       SEX  0.181124  0.181115  0.066096  2.740306 6.138196e-03
LTEMPPOLE  ~       SBP  0.001424  0.001424  0.092375  0.015415 9.877011e-01
LTEMPPOLE  ~       DBP  0.113416  0.113410  0.088396  1.283055 1.994729e-01
     TMTB  ~ LTEMPPO